<a href="https://colab.research.google.com/github/teju917/GENAITUTORIALS050/blob/main/program_9_multiagent_education.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Program 9: Multi-Agent System – How AI Is Transforming the Educational Field
**Tools used:** Groq API (LLM) + LangGraph (agent workflow) + Gradio (web interface)

**Agent pipeline:** Planner → Researcher → Analyst → Writer

In [1]:
!pip install -q groq langgraph

In [3]:
from groq import Groq
from google.colab import userdata

# Retrieve API key from Colab Secrets
api_key = userdata.get("GROQ_API_KEY")

# Initialize Groq client
client = Groq(api_key=api_key)

print("Groq API connected successfully!")

Groq API connected successfully!


In [4]:
# Use a currently supported Groq model
MODEL = "openai/gpt-oss-120b"

# Remove accidental spaces from API key
if client.api_key:
    client.api_key = client.api_key.strip()

# Test the model with an education-related question
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain in simple words how artificial intelligence is changing education."
        }
    ],
    temperature=0.7,
    max_completion_tokens=1024
)

# Print the generated response
print(response.choices[0].message.content)

**How AI is Changing Education – A Simple Overview**

| What AI Does | How It Looks in the Classroom | Why It Matters |
|--------------|------------------------------|----------------|
| **Personalizes Learning** | • AI‑powered platforms (like adaptive math apps) watch how a student solves problems and then give easier or harder tasks just for them.<br>• Homework recommendations match each learner’s pace and style. | Students get help that fits *their* needs, so they stay challenged but not overwhelmed. |
| **Offers Instant Help** | • Chatbots answer “I don’t understand this equation” 24/7.<br>• Voice assistants can read a paragraph aloud or explain a concept in plain language. | Learners can get clarification right away, without waiting for office hours. |
| **Creates Smarter Content** | • AI can turn a textbook chapter into a short video, an interactive quiz, or a visual diagram automatically.<br>• It can translate materials into other languages or add captions for accessibility. | T

In [5]:
def create_agent(role, task):

    prompt = f"""
    You are an AI agent working in the education domain.

    Your role: {role}

    Your responsibility:
    {task}

    Provide a clear, detailed and structured response.
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": "Perform your assigned task."}
        ],
        temperature=0.5
    )

    return response.choices[0].message.content

In [6]:
def planner_agent(user_task):

    return create_agent(
        role="Education Project Planner",
        task=f"""
        Understand the following education-related task:
        {user_task}

        Break it into clear steps.
        Identify the main objectives (students, teachers, institutions).
        Suggest how to approach the topic of AI in education.
        """
    )

In [7]:
def researcher_agent(plan):

    return create_agent(
        role="Education Technology Researcher",
        task=f"""
        Research the following plan:

        {plan}

        Explain relevant AI concepts used in education
        (personalised learning, intelligent tutors, automated grading, etc.),
        provide useful information and real-world examples,
        and identify important details.
        """
    )

In [8]:
def analyst_agent(research):

    return create_agent(
        role="Educational Data Analyst",
        task=f"""
        Analyse the following research:

        {research}

        Identify key findings,
        advantages, disadvantages,
        possible problems (privacy, bias, over-dependence on AI)
        and recommendations for schools and colleges.
        """
    )

In [9]:
def writer_agent(analysis):

    return create_agent(
        role="Academic Report Writer",
        task=f"""
        Create a final report using this analysis:

        {analysis}

        Include:
        1. Introduction
        2. Main findings
        3. Analysis
        4. Recommendations
        5. Conclusion

        Make the report clear and well structured.
        """
    )

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph imported successfully!")

LangGraph imported successfully!


In [11]:
class AgentState(TypedDict):

    user_task: str
    plan: str
    research: str
    analysis: str
    final_report: str

In [12]:
def planner_node(state: AgentState):

    result = planner_agent(state["user_task"])

    return {"plan": result}


def researcher_node(state: AgentState):

    result = researcher_agent(state["plan"])

    return {"research": result}


def analyst_node(state: AgentState):

    result = analyst_agent(state["research"])

    return {"analysis": result}


def writer_node(state: AgentState):

    result = writer_agent(state["analysis"])

    return {"final_report": result}

In [13]:
workflow = StateGraph(AgentState)

# Add agents as nodes
workflow.add_node("planner", planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("analyst", analyst_node)
workflow.add_node("writer", writer_node)

# Define execution order
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "researcher")
workflow.add_edge("researcher", "analyst")
workflow.add_edge("analyst", "writer")
workflow.add_edge("writer", END)

# Compile workflow
app = workflow.compile()

print("Multi-Agent Workflow created successfully!")

Multi-Agent Workflow created successfully!


## Running the Multi-Agent Workflow
The task below is passed through all four agents one after another.

In [14]:
user_input = """
Explain how Artificial Intelligence
is transforming the educational field.
Include benefits, challenges and future opportunities.
"""

initial_state = {
    "user_task": user_input,
    "plan": "",
    "research": "",
    "analysis": "",
    "final_report": ""
}

result = app.invoke(initial_state)

print("Multi-Agent Workflow Completed!")

Multi-Agent Workflow Completed!


In [15]:
print("\n========== PLANNER OUTPUT ==========\n")
print(result["plan"])

print("\n========== RESEARCHER OUTPUT ==========\n")
print(result["research"])

print("\n========== ANALYST OUTPUT ==========\n")
print(result["analysis"])

print("\n========== FINAL REPORT ==========\n")
print(result["final_report"])


========== PLANNER OUTPUT ==========

## 📚 Education Project Planner – “AI Transforming Education”

Below is a **complete, step‑by‑step plan** that you can hand‑off to a curriculum team, a professional‑development workshop, or a strategic‑planning committee.  
It explains **how Artificial Intelligence (AI) is reshaping the educational field**, outlines **benefits, challenges, and future opportunities**, identifies the **key objectives for each stakeholder group (students, teachers, institutions)**, and proposes a **structured approach** for presenting the topic.

---

### 1️⃣ Project Overview  

| Element | Description |
|---------|-------------|
| **Title** | *Artificial Intelligence in Education: Benefits, Challenges & Future Opportunities* |
| **Purpose** | Equip all education stakeholders with a deep, evidence‑based understanding of AI’s impact, so they can make informed decisions, adopt effective tools, and shape policy. |
| **Scope** | K‑12 and higher‑education contexts (can be 

In [16]:
user_task = input("Enter your task: ")

initial_state = {
    "user_task": user_task,
    "plan": "",
    "research": "",
    "analysis": "",
    "final_report": ""
}

result = app.invoke(initial_state)

print("\n========== FINAL AI REPORT ==========\n")
print(result["final_report"])

Enter your task: HOW AI IS TRNASFORMED IN EDUCTAIONAL FIED

========== FINAL AI REPORT ==========

**AI‑Powered Concepts that Are Reshaping Education (2024‑2025)**  
*Final Academic Report – Prepared for K‑12 Schools and Colleges/Universities*  

---  

## 1. Introduction  

Artificial‑intelligence (AI) has moved from experimental labs into everyday classrooms and campuses.  From adaptive learning platforms that tailor each lesson to a learner’s mastery level, to large‑language‑model (LLM) assistants that draft quizzes in seconds, AI now touches **instruction, assessment, student support, and administrative operations**.  

The purpose of this report is to:

1. **Summarise the most robust empirical findings** (2022‑2024) on nine high‑impact AI concepts.  
2. **Weigh benefits against the most frequently reported risks** (privacy, bias, over‑dependence, etc.).  
3. **Identify implementation‑level challenges** that schools and higher‑education institutions must address before scaling.  
4

In [17]:
!pip install -q gradio

In [18]:
import gradio as gr

def run_multi_agent(user_task):

    if not user_task or not user_task.strip():
        warning = "⚠️ Please enter a topic first."
        return warning, warning, warning, warning

    initial_state = {
        "user_task": user_task,
        "plan": "",
        "research": "",
        "analysis": "",
        "final_report": ""
    }

    result = app.invoke(initial_state)

    return (
        result["plan"],
        result["research"],
        result["analysis"],
        result["final_report"]
    )

In [ ]:
import gradio as gr

header_html = """
<div style="text-align:center; padding:28px 16px; border-radius:16px;
            background:linear-gradient(135deg,#0f2027,#203a43,#2c5364); color:white;">
    <h1 style="margin:0; font-size:2.2em;">🧑‍🏫 EduAI Lab</h1>
    <p style="margin:8px 0 0 0; font-size:1.1em; opacity:0.9;">
        A 4-agent AI team that explains how AI is transforming education
    </p>
</div>
"""

footer_html = """
<div style="text-align:center; padding:14px; margin-top:10px; opacity:0.75;">
    🗂️ Planner &rarr; 🔎 Researcher &rarr; 📈 Analyst &rarr; ✍️ Writer
    <br>Built with Groq + LangGraph + Gradio
</div>
"""

with gr.Blocks() as demo:

    gr.HTML(header_html)

    with gr.Row():

        # ---------- Left column : input ----------
        with gr.Column(scale=1):

            gr.Markdown("### 💬 Ask the Agents")

            user_input = gr.Textbox(
                label="Education topic",
                placeholder="e.g. How is AI transforming the educational field?",
                lines=5
            )

            submit_btn = gr.Button("▶️ Run Agent Team", variant="primary")
            clear_btn = gr.Button("🧹 Clear")

            gr.Examples(
                examples=[
                    "Explain how AI is transforming the educational field.",
                    "How does AI enable personalised learning for students?",
                    "What are the risks of AI tutors and AI grading in schools?",
                    "How will AI change the role of teachers in the next 10 years?",
                ],
                inputs=user_input,
                label="Try these topics"
            )

            gr.Markdown(
                "**Agent team**\n\n"
                "1. 🗂️ Planner – breaks the topic into steps\n"
                "2. 🔎 Researcher – gathers concepts & examples\n"
                "3. 📈 Analyst – findings, pros, cons\n"
                "4. ✍️ Writer – final structured report"
            )

        # ---------- Right column : tabbed output ----------
        with gr.Column(scale=2):

            gr.Markdown("### 📑 Agent Outputs")

            with gr.Tabs():
                with gr.TabItem("🗂️ Plan"):
                    plan_out = gr.Markdown()
                with gr.TabItem("🔎 Research"):
                    research_out = gr.Markdown()
                with gr.TabItem("📈 Analysis"):
                    analysis_out = gr.Markdown()
                with gr.TabItem("📝 Final Report"):
                    report_out = gr.Markdown()

    gr.HTML(footer_html)

    outputs_list = [plan_out, research_out, analysis_out, report_out]

    submit_btn.click(
        fn=run_multi_agent,
        inputs=user_input,
        outputs=outputs_list
    )

    clear_btn.click(
        fn=lambda: ("", "", "", "", ""),
        inputs=None,
        outputs=[user_input] + outputs_list
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://52dd2998c655828f90.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
